# 📊 Laporan Kualitas Data — ChatKasir

| | |
|---|---|
| **Author** | Muhammad Faradi Eka Damara (DS-1 — Data Engineer) |
| **Tim** | CC26-PSU065 |
| **Tanggal** | 30 April 2026 |
| **Deskripsi** | Laporan kualitas data awal dari tiga dataset yang digunakan dalam proyek ChatKasir: dataset makanan, dataset slang, dan dataset sintetis. |

---

## 📁 Dataset yang Dinilai

| No | Nama File | Sumber | Keterangan |
|---|---|---|---|
| 1 | `food_utama.csv` | `eriko-syah/indonesian-food` (HuggingFace) + `ariqsyahalam/indonesia-food-delivery-gofood-product-list` (Kaggle) | Gabungan dua sumber nama makanan Indonesia |
| 2 | `slang_utama.csv` | `nahiar/indonesia-slang` (HuggingFace) + `theonlydo/indonesia-slang` (HuggingFace) | Gabungan dua kamus slang Indonesia |
| 3 | `synthetic_orders_1000food_100000.csv` | Rule-Based Generation (DS-1) | Dataset sintetis 1000 produk unik, 100rb baris |

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print('✅ Library siap!')

---
## 📤 Upload Dataset
Jalankan cell di bawah lalu upload ketiga file:
- `food_utama.csv`
- `slang_utama.csv`
- `synthetic_orders_1000food_100000.csv`

In [ ]:
from google.colab import files
uploaded = files.upload()
print(f'\nFile yang diupload: {list(uploaded.keys())}')

---
## 1️⃣ Dataset Makanan — `food_utama.csv`

In [ ]:
df_food = pd.read_csv('food_utama.csv')

print('=' * 50)
print('ASSESSING food_utama.csv')
print('=' * 50)

print(f'\nJumlah baris  : {len(df_food)}')
print(f'Kolom         : {df_food.columns.tolist()}')
print(f'\nPreview:\n{df_food.head(10)}')

In [ ]:
# Cek null
print(f'Nilai kosong     : {df_food["name"].isnull().sum()}')

# Cek duplikat
print(f'Duplikat         : {df_food.duplicated().sum()}')

# Cek angka
print(f'Ada angka        : {df_food["name"].str.contains(r"[0-9]", regex=True).sum()} baris')

# Cek karakter spesial
print(f'Karakter spesial : {df_food["name"].str.contains(r"[^a-z\s]", regex=True).sum()} baris')

# Cek huruf kapital
print(f'Huruf kapital    : {df_food["name"].str.contains(r"[A-Z]", regex=True).sum()} baris')

# Cek spasi berlebih
print(f'Spasi berlebih   : {df_food["name"].str.contains(r"\s{2,}", regex=True).sum()} baris')

In [ ]:
# Distribusi panjang nama
df_food['panjang_kata'] = df_food['name'].str.split().str.len()
print('Distribusi panjang nama (kata):')
print(df_food['panjang_kata'].value_counts().sort_index())

print(f'\nNama dengan 1 kata (sample 10):')
print(df_food[df_food['panjang_kata'] == 1]['name'].head(10).tolist())

print(f'\nNama terpanjang (5 kata ke atas):')
print(df_food[df_food['panjang_kata'] >= 5]['name'].head(10).tolist())

In [ ]:
print('=' * 50)
print('CLEANING food_utama.csv')
print('=' * 50)

before = len(df_food)

# Fix spasi berlebih
df_food['name'] = df_food['name'].str.replace(r'\s{2,}', ' ', regex=True).str.strip()

# Hapus duplikat yang mungkin muncul setelah fix spasi
df_food = df_food.drop_duplicates(subset=['name']).reset_index(drop=True)

# Hapus kolom bantu
df_food = df_food.drop(columns=['panjang_kata'])

print(f'Sebelum cleaning : {before} baris')
print(f'Setelah cleaning : {len(df_food)} baris')
print(f'Dibuang          : {before - len(df_food)} baris')

print(f'\nVerifikasi final:')
print(f'  Nilai kosong     : {df_food["name"].isnull().sum()}')
print(f'  Duplikat         : {df_food.duplicated().sum()}')
print(f'  Spasi berlebih   : {df_food["name"].str.contains(r"\s{2,}", regex=True).sum()}')
print(f'  Total baris final: {len(df_food)}')

### 📝 Catatan Dataset Makanan
- Dataset sudah bersih sejak awal — tidak ada nilai kosong, duplikat, angka, karakter spesial, atau huruf kapital
- Mayoritas nama makanan terdiri dari **2–3 kata** (72.4% dari total)
- Nama makanan terpanjang mencapai **5 kata ke atas**

---
## 2️⃣ Dataset Slang — `slang_utama.csv`

In [ ]:
df_slang = pd.read_csv('slang_utama.csv')

print('=' * 50)
print('ASSESSING slang_utama.csv')
print('=' * 50)

print(f'\nJumlah baris  : {len(df_slang)}')
print(f'Kolom         : {df_slang.columns.tolist()}')
print(f'\nPreview:\n{df_slang.head(10)}')

In [ ]:
# Cek null
print(f'Nilai kosong slang   : {df_slang["slang"].isnull().sum()}')
print(f'Nilai kosong formal  : {df_slang["formal"].isnull().sum()}')

# Cek duplikat
print(f'Duplikat (full row)  : {df_slang.duplicated().sum()}')
print(f'Duplikat kolom slang : {df_slang["slang"].duplicated().sum()}')

# Cek huruf kapital
print(f'Huruf kapital slang  : {df_slang["slang"].str.contains(r"[A-Z]", regex=True).sum()} baris')
print(f'Huruf kapital formal : {df_slang["formal"].str.contains(r"[A-Z]", regex=True).sum()} baris')

# Cek spasi berlebih
print(f'Spasi berlebih slang  : {df_slang["slang"].str.contains(r"\s{2,}", regex=True).sum()} baris')
print(f'Spasi berlebih formal : {df_slang["formal"].str.contains(r"\s{2,}", regex=True).sum()} baris')

# Cek slang = formal
sama = (df_slang['slang'] == df_slang['formal'])
print(f'Slang sama dg formal : {sama.sum()} baris')

In [ ]:
# Panjang slang
df_slang['panjang_slang'] = df_slang['slang'].str.len()
print(f'Rata-rata panjang slang : {df_slang["panjang_slang"].mean():.1f} karakter')
print(f'Slang terpendek         : {df_slang["panjang_slang"].min()} karakter')
print(f'Slang terpanjang        : {df_slang["panjang_slang"].max()} karakter')

print(f'\nSlang 1 karakter (edge case):')
print(df_slang[df_slang['panjang_slang'] == 1][['slang', 'formal']].to_string(index=False))
print('\n⚠️ Catatan: Slang 1 karakter sangat pendek namun valid dalam konteks chat WhatsApp')

In [ ]:
print('=' * 50)
print('CLEANING slang_utama.csv')
print('=' * 50)

before = len(df_slang)

# Hapus slang = formal
df_slang = df_slang[df_slang['slang'] != df_slang['formal']].copy()
print(f'Hapus slang = formal : {before - len(df_slang)} baris dibuang')

# Fix spasi berlebih
df_slang['slang']  = df_slang['slang'].str.replace(r'\s{2,}', ' ', regex=True).str.strip()
df_slang['formal'] = df_slang['formal'].str.replace(r'\s{2,}', ' ', regex=True).str.strip()

# Hapus duplikat setelah cleaning
before2 = len(df_slang)
df_slang = df_slang.drop_duplicates(subset=['slang']).reset_index(drop=True)
print(f'Hapus duplikat       : {before2 - len(df_slang)} baris dibuang')

# Hapus kolom bantu
df_slang = df_slang.drop(columns=['panjang_slang'])

print(f'\nVerifikasi final:')
print(f'  Nilai kosong slang  : {df_slang["slang"].isnull().sum()}')
print(f'  Nilai kosong formal : {df_slang["formal"].isnull().sum()}')
print(f'  Duplikat            : {df_slang.duplicated().sum()}')
print(f'  Slang = formal      : {(df_slang["slang"] == df_slang["formal"]).sum()}')
print(f'  Total baris final   : {len(df_slang)}')

### 📝 Catatan Dataset Slang
- Dataset sudah bersih sejak awal — tidak ada nilai kosong, duplikat, atau inkonsistensi format
- Terdapat **2 slang 1 karakter** (`g` → tidak, `y` → iya) yang valid dalam konteks chat WhatsApp

---
## 3️⃣ Dataset Sintetis — `synthetic_orders_1000food_100000.csv`

In [ ]:
df_sint = pd.read_csv('synthetic_orders_1000food_100000.csv')

print('=' * 50)
print('ASSESSING synthetic_orders_1000food_100000.csv')
print('=' * 50)

print(f'\nTotal baris  : {len(df_sint)}')
print(f'Kolom        : {df_sint.columns.tolist()}')
print(f'\nNilai kosong:\n{df_sint.isnull().sum()}')

In [ ]:
pola_harga = {
    'rb (contoh: 15rb)'            : r'\d+rb',
    'k (contoh: 15k)'              : r'\d+k\b',
    'ribu (contoh: 15 ribu)'       : r'\d+ ribu',
    '.000 (contoh: 15.000)'        : r'\d+\.\d{3}',
    'rp...rb (contoh: rp15rb)'     : r'rp\d+rb',
    'rp ...000 (contoh: rp 15.000)': r'rp \d+\.\d{3}',
}
pola_gabungan = r'\d+rb|\d+k\b|\d+ ribu|\d+\.\d{3}|rp\d+rb|rp \d+\.\d{3}'

print('--- VALIDASI FORMAT HARGA ---')
for nama, pola in pola_harga.items():
    jumlah = df_sint['input_text'].str.contains(pola, regex=True).sum()
    persen = jumlah / len(df_sint) * 100
    print(f'  {nama:40s}: {jumlah:6} baris ({persen:.1f}%)')

tidak_ada_harga = ~df_sint['input_text'].str.contains(pola_gabungan, regex=True)
print(f'\n  Baris tanpa format harga apapun : {tidak_ada_harga.sum()}')
print(f'  Baris dengan price_satuan = -1  : {(df_sint["price_satuan"] == -1).sum()}')
print(f'  Selisih                         : {tidak_ada_harga.sum() - (df_sint["price_satuan"] == -1).sum()} baris ⚠️')

In [ ]:
pola_jumlah = {
    'angka saja (contoh: 3)'         : r'\b[1-9]\b',
    'angka + porsi (contoh: 3 porsi)': r'\d+ porsi',
    'angka + pcs (contoh: 3 pcs)'    : r'\d+ pcs',
    'angka + buah (contoh: 3 buah)'  : r'\d+ buah',
    'angka + bungkus'                : r'\d+ bungkus',
}

print('--- VALIDASI FORMAT JUMLAH ---')
for nama, pola in pola_jumlah.items():
    jumlah = df_sint['input_text'].str.contains(pola, regex=True).sum()
    persen = jumlah / len(df_sint) * 100
    print(f'  {nama:45s}: {jumlah:6} baris ({persen:.1f}%)')

print('\n⚠️ Catatan: Variasi satuan (porsi/pcs/buah/bungkus) belum di-generate — nice to have, bukan blocker')

In [ ]:
print('--- VALIDASI KOLOM quantity ---')
print(f'  Tipe data      : {df_sint["quantity"].dtype}')
print(f'  Nilai min      : {df_sint["quantity"].min()}')
print(f'  Nilai max      : {df_sint["quantity"].max()}')
print(f'  Distribusi:\n{df_sint["quantity"].value_counts().sort_index()}')

print('\n--- VALIDASI KOLOM price_satuan ---')
print(f'  Tipe data             : {df_sint["price_satuan"].dtype}')
print(f'  Baris price = -1      : {(df_sint["price_satuan"] == -1).sum()}')
print(f'  Baris price > 0       : {(df_sint["price_satuan"] > 0).sum()}')
print(f'  Baris price = 0       : {(df_sint["price_satuan"] == 0).sum()}')
print(f'  Harga min (selain -1) : {df_sint[df_sint["price_satuan"] > 0]["price_satuan"].min()}')
print(f'  Harga max             : {df_sint["price_satuan"].max()}')

In [ ]:
print('--- VALIDASI [SEP] ---')
ada_sep   = df_sint['input_text'].str.contains(r'\[SEP\]', regex=True).sum()
tidak_sep = (~df_sint['input_text'].str.contains(r'\[SEP\]', regex=True)).sum()
print(f'  Ada [SEP]    : {ada_sep} baris ({ada_sep/len(df_sint)*100:.1f}%)')
print(f'  Tanpa [SEP]  : {tidak_sep} baris ({tidak_sep/len(df_sint)*100:.1f}%)')
print('  ✅ Baris tanpa [SEP] adalah pattern 1 tanpa konfirmasi seller — disengaja')

print('\n--- PREVIEW 5 BARIS ACAK ---')
print(df_sint.sample(5, random_state=42)[['input_text', 'product', 'quantity', 'price_satuan', 'pattern']].to_string(index=False))

### 📝 Catatan Dataset Sintetis
- **17.500 baris** tidak memiliki harga eksplisit (`price_satuan = -1`) — disengaja untuk melatih model menangani kasus tanpa harga
- Selisih **84 baris** antara `baris_tanpa_harga_teks` dan `price_satuan = -1` — perlu investigasi lanjut
- **2.491 baris tanpa [SEP]** (2.5%) — disengaja, pattern 1 tanpa konfirmasi seller
- ⚠️ **Variasi satuan (porsi/pcs/buah/bungkus) belum ada** — dicatat sebagai item improvement
- ✅ Distribusi quantity **sangat merata** (1–10, ~10% per nilai)

---
## 4️⃣ Ringkasan Temuan & Status

In [ ]:
temuan = [
    {'Dataset': 'food_utama.csv',                         'Temuan': 'Tidak ada masalah ditemukan',                             'Severity': '✅ OK',    'Status': 'Selesai'},
    {'Dataset': 'slang_utama.csv',                        'Temuan': '2 slang 1 karakter (g→tidak, y→iya) — valid',            'Severity': '✅ OK',    'Status': 'Selesai'},
    {'Dataset': 'synthetic_orders_1000food_100000.csv',   'Temuan': 'Selisih 84 baris tanpa_harga_teks vs price=-1',           'Severity': '⚠️ Minor', 'Status': 'Perlu investigasi lanjut'},
    {'Dataset': 'synthetic_orders_1000food_100000.csv',   'Temuan': 'Variasi satuan (porsi/pcs/buah/bungkus) belum ada',       'Severity': '⚠️ Minor', 'Status': 'Nice to have — tambah jika akurasi model kurang'},
    {'Dataset': 'synthetic_orders_1000food_100000.csv',   'Temuan': '2.491 baris tanpa [SEP] (2.5%)',                          'Severity': '✅ OK',    'Status': 'Disengaja — pattern 1 tanpa konfirmasi seller'},
]

df_temuan = pd.DataFrame(temuan)
print('=== RINGKASAN TEMUAN ===')
print(df_temuan.to_string(index=False))

In [ ]:
# Ringkasan ukuran dataset final — dibaca langsung dari dataframe
ukuran = [
    {'Dataset': 'food_utama.csv',                       'Baris': len(df_food),  'Kolom': len(df_food.columns),  'Status': '✅ Final'},
    {'Dataset': 'slang_utama.csv',                      'Baris': len(df_slang), 'Kolom': len(df_slang.columns), 'Status': '✅ Final'},
    {'Dataset': 'synthetic_orders_1000food_100000.csv', 'Baris': len(df_sint),  'Kolom': len(df_sint.columns),  'Status': '✅ Final — digunakan AI-1 (Achmad Rif\'an)'},
]

df_ukuran = pd.DataFrame(ukuran)
print('=== UKURAN DATASET FINAL ===')
print(df_ukuran.to_string(index=False))
print('\n✅ Semua dataset telah diassess, dicleaning, dan siap digunakan untuk pelatihan model')